# Lab 01.5 – Integrated GPIO Mini-Project

This notebook contains an individual assignment that combines the concepts studied in Lab 01.1–01.4.

> **Important:** The hardware initialization is provided, but the application logic must be designed and implemented by the student.


## Learning objectives

After completing this assignment, you should be able to:
- combine switch and push-button inputs with LED outputs;
- use masks and bitwise operations on 4-bit values;
- detect a new button press and apply software debouncing;
- implement a finite-state application with several display modes;
- perform periodic actions without blocking button detection.


## Hardware initialization

Run this cell once. If the overlay is already loaded, running it again only reinitializes the GPIO objects.


In [ ]:
from pynq import Overlay
import time

ol = Overlay("base.bit")

switches = ol.switches_gpio.channel1
switches.setdirection("in")
switches.setlength(4)

buttons = ol.btns_gpio.channel1
buttons.setdirection("in")
buttons.setlength(4)

leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

leds.write(0, 0b1111)
print("GPIO initialized.")


# Task 5 – Interactive 4-bit LED Controller

Develop a Python application that controls the four LEDs using the switches and push buttons.

## Required behavior

1. When the program starts, read the four switches and use their binary value as the initial value of `value`.
2. Keep `value` in the interval **0…15**.
3. Detect only a **new press** of each button; holding a button must not repeat the command.
4. Use the following controls:
   - **BTN0** – increment `value`; after 15, continue from 0;
   - **BTN1** – decrement `value`; after 0, continue from 15;
   - **BTN2** – select the next display mode;
   - **BTN3** – stop the program and switch off all LEDs.
5. Implement the following display modes:

| Mode | Name | LED behavior |
|---:|---|---|
| 0 | NORMAL | Display `value` in binary |
| 1 | COMPLEMENT | Display the 4-bit complement of `value` |
| 2 | ROTATE | Circularly rotate the displayed pattern left every 0.25 s |
| 3 | BLINK | Alternate between `value` and 0 every 0.50 s |

6. Print the current value and mode only when either one changes. Example: `Value = 6 (0110), mode = COMPLEMENT`.
7. Button commands must remain responsive while ROTATE or BLINK is active. Do **not** use a long `time.sleep()` inside either mode.


## Design stage

Before writing the program, complete the table.

| Item | Your decision |
|---|---|
| Variables that describe the application state |  |
| Expression for increment with wrap-around |  |
| Expression for decrement with wrap-around |  |
| Expression for the 4-bit complement |  |
| Expression for a circular left rotation |  |
| Method used to detect a new button press |  |
| Method used to schedule ROTATE and BLINK |  |


## Implementation hints

- Use `& 0xF` whenever a result must be restricted to four bits.
- A new press can be detected by comparing `current_buttons` with `previous_buttons`.
- The current time can be read with `time.monotonic()`. Store the time of the previous periodic update and compare the difference with the required interval.
- In ROTATE mode, rotate a separate variable such as `display_value`; do not destroy the stored `value`.
- When the value or mode changes, reinitialize the timing and display variables needed by the new mode.
- Keep the main-loop delay short, for example `time.sleep(0.01)`.


## Student implementation

Complete the missing sections. You may add helper functions if they make the program clearer.


In [ ]:
MODE_NAMES = ["NORMAL", "COMPLEMENT", "ROTATE", "BLINK"]

value = switches.read() & 0xF
mode = 0
previous_buttons = 0
display_value = value
last_update = time.monotonic()
blink_on = True

# TODO 1: display the initial value and print the initial state.

raise NotImplementedError("Complete the TODO sections, then remove this line.")

while True:
    current_buttons = buttons.read() & 0xF

    # TODO 2: calculate which buttons have just been pressed.
    pressed = 0

    # TODO 3: implement BTN0, BTN1, BTN2 and BTN3.
    # Remember to reset display_value and timing information when required.

    now = time.monotonic()

    # TODO 4: calculate the LED pattern for all four modes.
    # ROTATE changes every 0.25 s; BLINK changes every 0.50 s.

    # TODO 5: write the resulting 4-bit pattern to the LEDs.

    previous_buttons = current_buttons
    time.sleep(0.01)

leds.write(0, 0b1111)
print("Program stopped.")


## Verification checklist

Demonstrate each test to the instructor.

- [ ] The initial switch value is displayed correctly.
- [ ] A held button generates only one command.
- [ ] Incrementing 15 produces 0.
- [ ] Decrementing 0 produces 15.
- [ ] BTN2 cycles through all four modes and returns to NORMAL.
- [ ] The complement is limited to four bits.
- [ ] ROTATE is circular and does not change the stored value.
- [ ] BLINK alternates at approximately 0.50 s.
- [ ] Buttons remain responsive in ROTATE and BLINK modes.
- [ ] BTN3 stops the loop and switches off all LEDs.


# Task 5+ – Automatic Counter with Adjustable Speed

Extend your Task 5 application with an **automatic counting mode**. Keep the four display modes from Task 5.

## Switch configuration

| Switch | Function |
|---|---|
| SW0 | `0` = manual counting, `1` = automatic counting |
| SW1 | `0` = count up, `1` = count down |
| SW3:SW2 | Select the automatic counting interval |

| SW3:SW2 | Interval |
|---:|---:|
| 00 | 1.00 s |
| 01 | 0.50 s |
| 10 | 0.25 s |
| 11 | 0.10 s |

## Button controls

- **BTN0** – start/pause automatic counting; in manual mode, increment once;
- **BTN1** – reset `value` to 0; in manual mode, decrement once;
- **BTN2** – select the next display mode;
- **BTN3** – stop the application and switch off all LEDs.

## Additional requirements

1. Read the switches continuously, so changing the direction or speed takes effect while the program is running.
2. Automatic counting, ROTATE and BLINK must use independent timing variables.
3. No functional delay may block the main loop. The only permitted fixed delay is the short polling delay.
4. Print a status line whenever the run/pause state, direction, interval, value or display mode changes.
5. The program must remain responsive even at the 0.10 s counting interval.


## Task 5+ suggested structure

Adapt your Task 5 program using the following plan:

1. Read and decode the switch configuration.
2. Detect new button presses and update the control state.
3. If automatic counting is enabled and running, update `value` when its interval expires.
4. Independently update the selected LED effect when its own interval expires.
5. Write the final pattern to the LEDs.

Recommended additional variables: `running`, `last_count_time`, `last_effect_time`, `count_interval`, and `direction`.


In [ ]:
# Implement Task 5+ here.
# Begin with your working Task 5 solution and extend it incrementally.

intervals = [1.00, 0.50, 0.25, 0.10]

# TODO: initialize the complete application state.
# TODO: implement the non-blocking main loop.


## Questions

1. Why is `& 0xF` useful in a 4-bit application?
2. What is the difference between a left shift and a circular left rotation?
3. Why can a mechanical push button generate several transitions during one physical press?
4. Why would a long `time.sleep()` make the application unresponsive?
5. Which variables define the state of your application?
6. In Task 5+, why are separate timers required for counting, rotating and blinking?
